## Import libraries

In [1]:
# Fundamentals
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
from glob import glob
import os, shutil
from tqdm import tqdm

# ObsPy
from obspy import read, read_inventory, UTCDateTime as utc
from obspy.geodetics.base import gps2dist_azimuth

import warnings
warnings.filterwarnings("ignore")

## Rotate and save seismograms

In [2]:
def rotate_ne_rt(n, e, ba):
    ba=np.radians(ba)
    r=-e*np.sin(ba)-n*np.cos(ba)
    t=-e*np.cos(ba)+n*np.sin(ba)
    return r, t

In [3]:
def find_orientation(code):
    sensor=orientation_table[orientation_table["Code"]==code].reset_index(drop=True)["Sensor"][0]
    if sensor=="HH":
        sub_table=orientation_table[orientation_table["Code"]==code].reset_index(drop=True)
        mis=sub_table["Misorientation"][0]
    elif sensor=="EL":
        sub_table=orientation_table[orientation_table["Code"]==code].reset_index(drop=True)
        mis=sub_table["Misorientation"][0]
    elif sensor=="HG":
        sub_table=orientation_table[orientation_table["Code"]==code].reset_index(drop=True)
        mis=sub_table["Misorientation"][0]
        
    return mis

In [4]:
evla, evlo=36.2823, 127.7608
eventdir_list=sorted(glob("/home/msseo/works/19.Korea_Directivity/49.251008_Okcheon/waveforms_100km/20*"))
for (count, eventdir) in tqdm(enumerate(eventdir_list)):
    selected_station_list=[]
    sacfile_list=sorted(glob(eventdir+"/*Z.sac"))
    orientation_table=pd.read_csv(f"/home/msseo/works/19.Korea_Directivity/49.251008_Okcheon/station_table/used_stations_100km_event{count+1}.csv", sep=',')
    
    for sacfile in sacfile_list:
        station_code=sacfile.split("/")[-1].split(".")[-3]
        selected_station_list.append(station_code)
        
    for code in selected_station_list:
        try:
            network=orientation_table[orientation_table["Code"]==code].reset_index(drop=True)["Network"][0]
            sensor=orientation_table[orientation_table["Code"]==code].reset_index(drop=True)["Sensor"][0]
            stla=orientation_table[orientation_table["Code"]==code].reset_index(drop=True)["Latitude"][0]
            stlo=orientation_table[orientation_table["Code"]==code].reset_index(drop=True)["Longitude"][0]
            yyyymmddhhmmss=eventdir.split("/")[-1]
            year, month, day=yyyymmddhhmmss[0:3], yyyymmddhhmmss[3:5], yyyymmddhhmmss[5:7]
            origin_time=f"{year}-{month}-{day}"
            baz=gps2dist_azimuth(evla, evlo, stla, stlo)[2]
            mis=find_orientation(code)

            tr_Z=read(f"/home/msseo/works/19.Korea_Directivity/49.251008_Okcheon/waveforms_100km/{yyyymmddhhmmss}/{yyyymmddhhmmss}.{network}.{code}.{sensor}Z.sac")[0]
            tr_N=read(f"/home/msseo/works/19.Korea_Directivity/49.251008_Okcheon/waveforms_100km/{yyyymmddhhmmss}/{yyyymmddhhmmss}.{network}.{code}.{sensor}N.sac")[0]
            tr_E=read(f"/home/msseo/works/19.Korea_Directivity/49.251008_Okcheon/waveforms_100km/{yyyymmddhhmmss}/{yyyymmddhhmmss}.{network}.{code}.{sensor}E.sac")[0]

            # Slice the traces from -100 s to 300 s relative to the origin time so that the lengths of traces are same
            t_before, t_after=50, 200
            tr_Z_origin=tr_Z.stats.starttime-tr_Z.stats.sac.b

            tr_Z_slice=tr_Z.copy().detrend("demean").taper(0.05).slice(tr_Z_origin-t_before, tr_Z_origin+t_after).detrend("demean").taper(0.05)
            tr_N_slice=tr_N.copy().detrend("demean").taper(0.05).slice(tr_Z_origin-t_before, tr_Z_origin+t_after).detrend("demean").taper(0.05)
            tr_E_slice=tr_E.copy().detrend("demean").taper(0.05).slice(tr_Z_origin-t_before, tr_Z_origin+t_after).detrend("demean").taper(0.05)

            n, e=tr_N_slice.data, tr_E_slice.data
            r, t=rotate_ne_rt(n, e, ba=(baz-mis)%360)
            tr_N_slice.data, tr_E_slice.data=r, t
            tr_Z_slice.stats.channel, tr_N_slice.stats.channel, tr_E_slice.stats.channel=f"{sensor}Z", f"{sensor}R", f"{sensor}T"

            os.makedirs(f"/home/msseo/works/19.Korea_Directivity/49.251008_Okcheon/rotated/{yyyymmddhhmmss}", exist_ok=True)
            tr_Z_slice.write(f"/home/msseo/works/19.Korea_Directivity/49.251008_Okcheon/rotated/{yyyymmddhhmmss}/{yyyymmddhhmmss}.{network}.{code}.{sensor}Z.sac", format="SAC")
            tr_N_slice.write(f"/home/msseo/works/19.Korea_Directivity/49.251008_Okcheon/rotated/{yyyymmddhhmmss}/{yyyymmddhhmmss}.{network}.{code}.{sensor}R.sac", format="SAC")
            tr_E_slice.write(f"/home/msseo/works/19.Korea_Directivity/49.251008_Okcheon/rotated/{yyyymmddhhmmss}/{yyyymmddhhmmss}.{network}.{code}.{sensor}T.sac", format="SAC")
        
        except:
            print(code)

12it [00:11,  1.03it/s]
